-- Job ID: rmisqlfactory_traffic_YYYYMMDDHHMMSS
```
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
```


# RMI Sample Queries: Traffic Operations Manager (GA)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/n0531m/rmi-sql-factory/blob/main/rmi-sql/assets/notebooks/ga/Traffic_Operations_Manager_Samples.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fn0531m%2Frmi-sql-factory%2Fmain%2Frmi-sql%2Fassets%2Fnotebooks%2Fga%2FTraffic_Operations_Manager_Samples.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/n0531m/rmi-sql-factory/blob/main/rmi-sql/assets/notebooks/ga/Traffic_Operations_Manager_Samples.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/n0531m/rmi-sql-factory/blob/main/rmi-sql/assets/notebooks/ga/Traffic_Operations_Manager_Samples.ipynb">
      <img width="32px" src="https://upload.wikimedia.org/wikipedia/commons/9/91/Octicons-mark-github.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>



This notebook contains sample queries for the **Traffic Operations Manager** persona, specifically for the **GA** stage.

## 1. Setup

In [ ]:
from google.colab import auth
import pandas as pd

auth.authenticate_user()

project_id = 'your-project-id' #@param {type:"string"}

### Writable Dataset

Several queries in this notebook (e.g., those creating Materialized Views, Models, or Views) require a **writable dataset** within your own project. 
**Note**: The source `LINKED_DATASET_NAME` dataset is a read-only subscription and cannot be used to store new resources.

Run the cell below to create a new dataset (e.g., `rmi_analysis`) in your project if you haven't already.

**Important**: When running queries that create new BigQuery resources (e.g., tables, views, models) outside of these `%%bigquery` magic cells, remember to manually prepend the job ID with `msqlfactory--` for proper tracking. For example: `bq query --job_id=msqlfactory--your-descriptive-job-name ...`

In [ ]:
dataset_id = "rmi_analysis" #@param {type:"string"}
! bq --location=US mk --dataset {project_id}:{dataset_id}

## GA (General Availability)

### tom1_peak_hour_delay.sql
**Business Question**: What is the average travel time delay during the morning peak (7-9 AM) for the top 10 most congested routes?
**Use Case**: Identifies critical morning commute bottlenecks to inform operational decisions or public messaging.


In [ ]:
%%bigquery --project {project_id} df_tom1_peak_hour_delay
/*
  ANALYTICAL PATTERN: Temporal Filtering
  This query uses EXTRACT(HOUR...) on a converted DATETIME to focus on local 
  Boston peak windows. It filters for active routes and applies a quality 
  check to ensure the geometry is a single ST_LineString.
*/

WITH peak_hour_data AS (
  SELECT
    h.selected_route_id,
    h.display_name,
    -- delay_ratio > 1.0 indicates travel time is slower than free-flow (static)
    SAFE_DIVIDE(h.duration_in_seconds, h.static_duration_in_seconds) AS delay_ratio
  FROM `LINKED_DATASET_NAME.historical_travel_time` AS h
  JOIN `LINKED_DATASET_NAME.routes_status` AS s USING (selected_route_id)
  WHERE h.record_time BETWEEN '2026-06-01' AND '2026-06-30'
    -- STATUS_RUNNING ensures we only analyze routes that are currently being monitored
    AND s.status = 'STATUS_RUNNING'
    -- AM Peak Window: 7:00 AM to 8:59 AM Local Time
    AND EXTRACT(HOUR FROM DATETIME(h.record_time, 'America/New_York')) BETWEEN 7 AND 8
    -- Geometry Integrity: Only process continuous, healthy paths (ST_LineString)
    AND ST_GEOMETRYTYPE(h.route_geometry) = 'ST_LineString'
    AND h.duration_in_seconds IS NOT NULL
    AND h.static_duration_in_seconds > 0
)
SELECT
  display_name,
  ROUND(AVG(delay_ratio), 3) AS avg_delay_ratio,
  COUNT(*) AS sample_count
FROM peak_hour_data
GROUP BY 1
-- Threshold: Filter for routes that are at least marginally slower than free-flow
HAVING avg_delay_ratio > 1.0
ORDER BY avg_delay_ratio DESC
LIMIT 10;


### tom2_persistent_bottlenecks.sql
**Business Question**: Which road segments (SRIs) have been in a 'TRAFFIC_JAM' state most frequently?
**Use Case**: Locates recurring local bottlenecks within routes, enabling targeted infrastructure investigation or signal timing adjustments.


In [ ]:
%%bigquery --project {project_id} df_tom2_persistent_bottlenecks
/*
  ANALYTICAL PATTERN: SRI Unnesting
  RMI routes store segment-level traffic states (SRI) in a nested array. 
  This query 'explodes' that array using UNNEST to audit the frequency of 
  severe congestion across the entire network.
*/

WITH exploded_sris AS (
  SELECT
    selected_route_id,
    display_name,
    -- 'speed' represents the RMI traffic state for that specific interval
    sri.speed
  FROM `LINKED_DATASET_NAME.recent_roads_data`,
  UNNEST(speed_reading_intervals) AS sri
  WHERE ST_GEOMETRYTYPE(route_geometry) = 'ST_LineString' AND record_time BETWEEN '2026-06-01' AND '2026-06-30'
)
SELECT
  selected_route_id,
  display_name,
  -- We count every occurrence of an interval being in a 'TRAFFIC_JAM'
  COUNT(*) AS traffic_jam_count
FROM exploded_sris
-- Filter exclusively for the most severe RMI congestion state
WHERE speed = 'TRAFFIC_JAM'
GROUP BY 1, 2
ORDER BY traffic_jam_count DESC
LIMIT 10;


### tom3_operational_health.sql
**Business Question**: Which active routes are currently flagged with a 'LOW_ROAD_USAGE' validation error?
**Use Case**: Monitors the reliability of data collection. Low usage flags indicate that insights for these routes may be based on fewer probes, requiring a review of route priority or placement.


In [ ]:
%%bigquery --project {project_id} df_tom3_operational_health
/*
  ANALYTICAL PATTERN: Status Auditing
  This query inspects the management plane table (routes_status) to identify 
  active routes that have quality warnings. This is critical for maintaining 
  trust in downstream traffic analytics.
*/

SELECT
  display_name,
  selected_route_id,
  status,
  validation_error,
  -- 'low_road_usage_start_time' is specifically populated when probe density drops below threshold
  low_road_usage_start_time,
  -- Time elapsed since the error was first detected (relative to sample end date)
  DATETIME_DIFF(DATETIME('2026-06-30'), DATETIME(low_road_usage_start_time, 'UTC'), DAY) AS days_in_error_state
FROM `LINKED_DATASET_NAME.routes_status`
-- We only care about errors on routes that are supposed to be active (STATUS_RUNNING)
WHERE status = 'STATUS_RUNNING'
  -- Filter specifically for the Low Road Usage warning
  AND validation_error = 'VALIDATION_ERROR_LOW_ROAD_USAGE'
ORDER BY days_in_error_state DESC;


### tom4_data_latency.sql
**Business Question**: Are there any active routes that have stopped sending data near the end of the snapshot period?
**Use Case**: Detects localized data gaps or "silent" routes in real-time, enabling operators to investigate issues before they impact reporting.


In [ ]:
%%bigquery --project {project_id} df_tom4_data_latency
/*
  ANALYTICAL PATTERN: Freshness Monitoring
  By comparing the max record_time per route against the overall dataset 
  end-time, we can identify routes that have 'gone silent'.
*/

WITH last_data_arrival AS (
  SELECT
    selected_route_id,
    -- Get the latest record timestamp for every route in the dataset
    MAX(record_time) AS last_arrival
  FROM `LINKED_DATASET_NAME.historical_travel_time`
  -- Focused partition scan for the full sample month
  WHERE ST_GEOMETRYTYPE(route_geometry) = 'ST_LineString' AND record_time BETWEEN '2026-06-01' AND '2026-06-30'
  GROUP BY 1
)
SELECT
  s.selected_route_id,
  s.display_name,
  l.last_arrival,
  -- Measured relative to the end of the sample period
  TIMESTAMP_DIFF(TIMESTAMP('2026-06-30 00:00:00'), l.last_arrival, MINUTE) as minutes_of_silence
FROM `LINKED_DATASET_NAME.routes_status` s
LEFT JOIN last_data_arrival l USING (selected_route_id)
-- Focus on routes that are supposed to be producing data
WHERE s.status = 'STATUS_RUNNING'
  -- Threshold: Highlight routes that haven't sent a record in the last few hours of the dataset
  AND l.last_arrival < TIMESTAMP('2026-06-29 21:55:00')
ORDER BY minutes_of_silence DESC;


### tom5_significant_event_detection.sql
**Business Question**: Which routes experienced a travel time more than double their static baseline in the last 24 hours?
**Use Case**: Automates the detection of extreme traffic events (accidents, severe weather, gridlock) that require immediate operational intervention.


In [ ]:
%%bigquery --project {project_id} df_tom5_significant_event_detection
/*
  ANALYTICAL PATTERN: Threshold-Based Alerting
  This query identifies major traffic incidents by flagging records where the 
  actual travel time is at least 2x the free-flow baseline (static_duration). 
  It applies quality filters to ensure alerts are only triggered for single, 
  continuous paths.
*/

SELECT
  h.display_name,
  h.selected_route_id,
  h.record_time,
  h.duration_in_seconds,
  h.static_duration_in_seconds,
  -- Delay ratio > 2.0 means travel time is 2x slower than ideal
  ROUND(SAFE_DIVIDE(h.duration_in_seconds, h.static_duration_in_seconds), 2) AS delay_ratio
FROM `LINKED_DATASET_NAME.historical_travel_time` AS h
JOIN `LINKED_DATASET_NAME.routes_status` AS s USING(selected_route_id)
-- Filter for the final days of the sample dataset
WHERE h.record_time BETWEEN '2026-06-27' AND '2026-06-30'
  -- Focus on active monitoring fleet
  AND s.status = 'STATUS_RUNNING'
  -- Filter for \"Significant\" events
  AND SAFE_DIVIDE(h.duration_in_seconds, h.static_duration_in_seconds) > 2.0
  -- Quality filter: Exclude non-continuous geometries
  AND ST_GEOMETRYTYPE(h.route_geometry) = 'ST_LineString'
  AND h.duration_in_seconds IS NOT NULL
  AND h.static_duration_in_seconds > 0
  ORDER BY delay_ratio DESC;


### tom6_dynamic_detour_detection.sql
**Business Question**: Which routes experienced dynamic rerouting/detours, and what was the travel time impact between the primary path and detour variations?
**Use Case**: Identifies recurring dynamic route variations or incident diversions caused by congestion, road closures, or routing engine shifts.


In [ ]:
%%bigquery --project {project_id} df_tom6_dynamic_detour_detection
/*
  ANALYTICAL PATTERN: Dynamic Path Variation & Detour Analysis
  
  IMPORTANT SCHEMA NOTE:
  The 'road_segment_ids' column (ARRAY<STRING> containing topological Place IDs)
  was added to 'historical_travel_time' on June 19, 2026.
  Telemetry records prior to this date have empty/NULL arrays for this column.
  Therefore, queries utilizing 'road_segment_ids' MUST filter for:
    record_time >= '2026-06-19' AND ARRAY_LENGTH(road_segment_ids) > 0
  
  WORKFLOW:
  1. Flattens road_segment_ids into a pipe-delimited string (fingerprint) to allow GROUP BY.
  2. Computes the frequency and average duration of each distinct physical path variant.
  3. Aggregates variations into an array sorted by frequency (Index 0 = Primary, Index 1+ = Detour).
  4. Computes the Detour Delay Multiplier to quantify the operational travel time penalty.
*/

WITH path_variants AS (
  SELECT
    h.selected_route_id,
    h.display_name,
    ARRAY_TO_STRING(h.road_segment_ids, '|') AS segment_ids,
    AVG(h.duration_in_seconds) AS avg_duration_in_seconds,
    COUNT(h.record_time) AS num_of_records
  FROM `LINKED_DATASET_NAME.historical_travel_time` AS h
  JOIN `LINKED_DATASET_NAME.routes_status` AS s USING (selected_route_id)
  -- Schema Temporal Filter: road_segment_ids is only populated from 2026-06-19 onward
  WHERE h.record_time >= '2026-06-19'
    AND ARRAY_LENGTH(h.road_segment_ids) > 0
    AND s.status = 'STATUS_RUNNING'
    AND h.duration_in_seconds IS NOT NULL
    AND h.static_duration_in_seconds > 0
  GROUP BY 1, 2, 3
),
ranked_routes AS (
  SELECT
    selected_route_id,
    display_name,
    ARRAY_AGG(
      STRUCT(segment_ids, avg_duration_in_seconds, num_of_records)
      ORDER BY num_of_records DESC
    ) AS variations
  FROM path_variants
  GROUP BY 1, 2
)
SELECT
  selected_route_id,
  display_name,
  ARRAY_LENGTH(variations) AS total_path_variations,
  variations[OFFSET(0)].num_of_records AS primary_path_samples,
  ROUND(variations[OFFSET(0)].avg_duration_in_seconds, 1) AS primary_avg_duration_seconds,
  variations[OFFSET(1)].num_of_records AS detour_path_samples,
  ROUND(variations[OFFSET(1)].avg_duration_in_seconds, 1) AS detour_avg_duration_seconds,
  -- Detour Delay Multiplier: > 1.0 means the detour was slower than the standard path
  ROUND(SAFE_DIVIDE(variations[OFFSET(1)].avg_duration_in_seconds, variations[OFFSET(0)].avg_duration_in_seconds), 2) AS detour_delay_multiplier
FROM ranked_routes
WHERE ARRAY_LENGTH(variations) > 1
ORDER BY detour_delay_multiplier DESC;
